# Pipeline ARIMA & LSTM v6 -- VERSI DIPERBAIKI (dengan Dokumentasi BEFORE/AFTER)

Notebook ini adalah **versi perbaikan (v6)** dari pipeline final v5 yang sebelumnya
dipakai Laura. Seluruh struktur, urutan eksekusi, dan mekanisme BEFORE/AFTER **tetap
sama** dengan v5 -- yang berubah **hanya dua nilai konstanta** pada Sel 0, berdasarkan
hasil eksperimen tambahan yang baru selesai dijalankan:

| Parameter | Nilai v5 (lama) | Nilai v6 (baru) | Alasan Perubahan |
|---|---|---|---|
| `CAP_FACTOR_ARIMA` | 1.0 | **0.8** | Eksperimen CAP_FACTOR_ARIMA (0.8 / 1.0 / 1.2 / 1.5) menunjukkan MAE ARIMA terendah justru di 0.8 (Rp2.408.274) dibanding 1.0 (Rp2.467.634) -- turun ~2.4%, tanpa mengubah MAE LSTM sama sekali. |
| `CAP_FACTOR_LSTM` | 1.5 | **1.0** | Eksperimen CAP_FACTOR_LSTM (1.0 s.d. 2.5) menunjukkan MAE dan RMSE LSTM **identik** pada seluruh nilai yang diuji (Rp1.327.530 / Rp1.587.740). Karena tidak ada perbedaan akurasi, dipilih batas paling ketat (1.0x nilai maksimum historis) sebagai *fail-safe* tambahan agar prediksi LSTM tidak pernah melebihi rekor penjualan tertinggi historis produk tersebut. |

Dua parameter lain yang sempat diuji **tidak diubah** dari v5, karena hasil eksperimen
justru mengonfirmasi konfigurasi v5 sudah tepat:

- **Grid Search AIC vs Order Tetap** -- Ablasi menunjukkan grid search AIC (12 order,
  MAE=Rp2.467.634) tetap mengungguli seluruh skenario order tetap yang diuji, termasuk
  order tetap (1,1,1) yang merupakan skenario terbaik kedua (MAE=Rp2.452.351). Grid
  search dipertahankan meski butuh ±1.992 kali lebih banyak proses fitting SARIMAX,
  karena selisih akurasi (~0.6%) dinilai berharga dan waktu total tetap jauh di bawah
  batas KNF-01 (49.7 detik untuk 166 produk).
- **RANDOM_SEED=42** -- Uji stabilitas lintas 5 seed menunjukkan CV LSTM sebesar 42.9%
  (wajar untuk model deep learning), namun MAE pada seed=42 (Rp1.327.530) berada dalam
  rentang median kelima seed yang diuji, sehingga tetap dipertahankan sebagai nilai
  konvensional yang representatif dan tidak diganti ke seed dengan MAE terendah
  (seed=0) demi konsistensi dengan seluruh dokumentasi historis proyek.

Seluruh sel BEFORE/AFTER pada notebook ini tetap mencetak baris:
```
[BEFORE] ... -> [AFTER] ...   (selisih: ...)
```
agar setiap angka pada Bab 4 skripsi tetap dapat ditelusuri sumbernya.

## Sel 0 -- Setup Environment dan Konstanta (v6: CAP_FACTOR_ARIMA & CAP_FACTOR_LSTM diperbarui)

In [1]:
import os, random, time, tracemalloc, warnings
from collections import Counter
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
os.environ.setdefault('PYTHONHASHSEED', '0')

import pandas as pd
import numpy as np
import tensorflow as tf

tf.config.threading.set_inter_op_parallelism_threads(1)
tf.config.threading.set_intra_op_parallelism_threads(1)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.arima.model import ARIMA
from joblib import Parallel, delayed

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

csv_path = "dataset_toko.csv"

# ============================== KONSTANTA ==============================
# Parameter di bawah ini adalah hasil akhir seluruh eksperimen (Bab 4.6 skripsi).
# Baris bertanda [v6] adalah nilai yang DIPERBARUI dari versi v5 sebelumnya.
SPLIT_PCT         = 0.80
MIN_BULAN         = 15
N_WINDOW_ARIMA    = 24
RANDOM_SEED       = 42
SEQ_LEN           = 6
N_CLUSTER         = 5
BIAS_HOLDOUT      = 7
CAP_FACTOR_ARIMA  = 0.8   # [v6] sebelumnya 1.0 -- MAE ARIMA turun ~2.4% (eksperimen CAP_FACTOR_ARIMA)
CAP_FACTOR_LSTM   = 1.0   # [v6] sebelumnya 1.5 -- MAE/RMSE LSTM identik di semua nilai; 1.0 dipilih sbg fail-safe paling ketat
MIN_BULAN_AKTIF   = 3
IQR_MULTIPLIER    = 1.5
TOP_N             = 10
TOLERANSI_KETAT   = 20
TOLERANSI_LONGGAR = 60
N_JOBS            = -1
ARIMA_MAXITER     = 100

ARIMA_ORDERS = [
    (1,1,1),(1,1,0),(0,1,1),(2,1,0),
    (0,1,2),(2,1,1),(1,1,2),(3,1,0),
    (0,1,3),(2,1,2),(1,2,1),(0,2,1),
]

KALENDER_LIBUR = {
    '2020-05':0.40,'2021-05':0.45,'2022-05':0.55,'2023-04':0.50,'2024-04':0.65,'2025-03':0.90,
    '2020-06':0.60,'2021-06':0.60,'2022-06':0.65,'2020-07':0.60,'2021-07':0.65,'2022-07':0.70,
    '2023-06':0.65,'2024-05':0.60,'2024-06':0.12,'2024-07':0.28,'2024-08':0.55,'2024-12':0.40,'2025-01':0.58,
}
BULAN_EKSKLUDE = ['2024-06','2024-07']
BULAN_ANOMALI  = BULAN_EKSKLUDE

FEATURES = ['lag1_r','lag2_r','lag3_r','lag6_r','lag12_r',
            'roll3_r','roll6_r','tren_3m','bulan','produk_id','faktor_libur']

def set_seed_ulang(offset=0):
    random.seed(RANDOM_SEED + offset)
    np.random.seed(RANDOM_SEED + offset)
    tf.random.set_seed(RANDOM_SEED + offset)

set_seed_ulang(0)
print("Setup selesai (v6).")
print(f"[v6] CAP_FACTOR_ARIMA = {CAP_FACTOR_ARIMA}  (sebelumnya 1.0 pada v5)")
print(f"[v6] CAP_FACTOR_LSTM  = {CAP_FACTOR_LSTM}  (sebelumnya 1.5 pada v5)")


Setup selesai (v6).
[v6] CAP_FACTOR_ARIMA = 0.8  (sebelumnya 1.0 pada v5)
[v6] CAP_FACTOR_LSTM  = 1.0  (sebelumnya 1.5 pada v5)


## Sel 1--3 -- Fungsi Helper (Momentum, Training/Prediksi ARIMA Paralel, Prediksi LSTM Batch)

Fungsi-fungsi ini **tidak diubah logikanya** dari v5. Karena `CAP_FACTOR_ARIMA` dan
`CAP_FACTOR_LSTM` dibaca sebagai variabel global dari Sel 0, perubahan nilai pada Sel 0
otomatis diterapkan di sini tanpa perlu mengubah kode fungsi.

In [2]:
def hitung_momentum(df_c):
    if len(df_c) < 3:
        return 1.0
    vals = df_c['qty'].values[-3:]
    if vals[0] > 0:
        tren = (vals[-1] - vals[0]) / vals[0]
        return float(np.clip(1.0 + tren * 0.10, 0.90, 1.10))
    return 1.0

def _latih_satu_produk_arima(produk, monthly_train):
    df_c = monthly_train[monthly_train['Nama Produk'] == produk].sort_values('bulan_period')
    df_c = df_c[~df_c['bulan_period'].astype(str).isin(BULAN_EKSKLUDE)].tail(N_WINDOW_ARIMA)
    if len(df_c) < 10:
        return produk, None, 1.0
    fl_v = df_c['bulan_period'].astype(str).map(lambda x: KALENDER_LIBUR.get(x, 1.0)).values
    ts_log = np.log1p(df_c['qty'].clip(lower=0.1).values / np.maximum(fl_v, 0.1))
    best_aic, best_order = np.inf, (1, 1, 1)
    for order in ARIMA_ORDERS:
        try:
            m = ARIMA(ts_log, order=order, enforce_stationarity=True, enforce_invertibility=True
                      ).fit(method_kwargs={'maxiter': ARIMA_MAXITER})
            if m.aic < best_aic:
                best_aic, best_order = m.aic, order
        except Exception:
            pass
    bias = 1.0
    if len(df_c) > BIAS_HOLDOUT + 8:
        try:
            tr_b, ho_b = df_c.iloc[:-BIAS_HOLDOUT], df_c.iloc[-BIAS_HOLDOUT:]
            ts_tb = np.log1p(tr_b['qty'].clip(lower=0.1).values / np.maximum(
                tr_b['bulan_period'].astype(str).map(lambda x: KALENDER_LIBUR.get(x, 1.0)).values, 0.1))
            m_b = ARIMA(ts_tb, order=best_order, enforce_stationarity=True, enforce_invertibility=True
                        ).fit(method_kwargs={'maxiter': ARIMA_MAXITER})
            fc_log = np.clip(m_b.forecast(steps=BIAS_HOLDOUT), -2.0, 10.0)
            fl_hb = ho_b['bulan_period'].astype(str).map(lambda x: KALENDER_LIBUR.get(x, 1.0)).values
            fc_qty = np.expm1(fc_log) * fl_hb
            rasio = max(float(np.asarray(fc_qty).sum()), 1e-6) / float(ho_b['qty'].values.sum())
            bias = float(np.clip(rasio, 0.7, 1.5))
        except Exception:
            bias = 1.0
    return produk, best_order, bias

def latih_semua_arima(produk_arima_l, monthly_train):
    hasil_paralel = Parallel(n_jobs=N_JOBS, backend='loky')(
        delayed(_latih_satu_produk_arima)(produk, monthly_train) for produk in produk_arima_l
    )
    best_orders_arima, bias_correction = {}, {}
    for produk, order, bias in hasil_paralel:
        bias_correction[produk] = bias
        if order is not None:
            best_orders_arima[produk] = order
    return best_orders_arima, bias_correction

def _prediksi_satu_produk_arima(produk, bulan_str, faktor_libur, monthly_avail,
                                 best_orders_arima, bias_correction, harga_rata2, max_qty_produk):
    df_p = monthly_avail[monthly_avail['Nama Produk'] == produk].sort_values('bulan_period')
    df_c = df_p[~df_p['bulan_period'].astype(str).isin(BULAN_EKSKLUDE)]
    if len(df_c) > N_WINDOW_ARIMA:
        df_c = df_c.tail(N_WINDOW_ARIMA)
    fc = None
    if produk in best_orders_arima and len(df_c) >= 6:
        try:
            qty_vals = df_c['qty'].clip(lower=0.1).values
            fl_vals = df_c['bulan_period'].astype(str).map(lambda x: KALENDER_LIBUR.get(x, 1.0)).values
            ts_log = np.log1p(qty_vals / np.maximum(fl_vals, 0.1))
            m = ARIMA(ts_log, order=best_orders_arima[produk],
                      enforce_stationarity=True, enforce_invertibility=True
                      ).fit(method_kwargs={'maxiter': ARIMA_MAXITER})
            fc_result = m.forecast(steps=1)
            fc_val = fc_result.iloc[0] if hasattr(fc_result, 'iloc') else np.asarray(fc_result).reshape(-1)[0]
            fc = max(0.0, float(np.expm1(float(np.clip(fc_val, -2.0, 10.0)))) * faktor_libur)
            bias = bias_correction.get(produk, 1.0)
            if bias > 0: fc = fc / bias
            fc = fc * hitung_momentum(df_c)
            # [v6] CAP_FACTOR_ARIMA sekarang 0.8 (sebelumnya 1.0)
            fc = min(max(0.0, fc), max_qty_produk.get(produk, fc) * CAP_FACTOR_ARIMA)
        except Exception:
            fc = None
    if fc is None and len(df_c) >= 1:
        recent = df_c['qty'].values[-min(6, len(df_c)):]
        bobot = np.array([1,2,3,4,5,6][-len(recent):], dtype=float)
        fc = float(np.average(recent, weights=bobot)) * faktor_libur
    if fc is None:
        fc = float(df_c['qty'].median()) * faktor_libur if len(df_c) > 0 else 0.0
    pred_qty = max(0.0, fc)
    return {'nama_produk': produk, 'pred_qty_arima': round(pred_qty, 2),
            'pred_rev_arima': round(pred_qty * harga_rata2.get(produk, 0), 0)}

def prediksi_arima(bulan_pred, monthly_avail, produk_layak, best_orders_arima,
                    bias_correction, harga_rata2, max_qty_produk, paralel=True):
    bulan_str = str(bulan_pred)
    faktor_libur = KALENDER_LIBUR.get(bulan_str, 1.0)
    if paralel:
        hasil = Parallel(n_jobs=N_JOBS, backend='loky')(
            delayed(_prediksi_satu_produk_arima)(
                produk, bulan_str, faktor_libur, monthly_avail, best_orders_arima,
                bias_correction, harga_rata2, max_qty_produk
            ) for produk in produk_layak
        )
    else:
        hasil = [_prediksi_satu_produk_arima(
            produk, bulan_str, faktor_libur, monthly_avail, best_orders_arima,
            bias_correction, harga_rata2, max_qty_produk
        ) for produk in produk_layak]
    return pd.DataFrame(hasil) if hasil else pd.DataFrame(columns=['nama_produk','pred_qty_arima','pred_rev_arima'])

def prediksi_lstm(bulan_pred, monthly_avail, produk_layak, cluster_models, cluster_scalers, cluster_bias,
                   produk_cluster, le, harga_rata2, median_qty_produk, max_qty_produk):
    bulan_str = str(bulan_pred)
    faktor_libur = KALENDER_LIBUR.get(bulan_str, 1.0)
    bulan_int = bulan_pred.month
    seq_per_cluster = {cid: {'produk': [], 'X': []} for cid in cluster_models.keys()}
    for produk in produk_layak:
        if produk not in le.classes_: continue
        cid = produk_cluster.get(produk, 0)
        if cid not in cluster_models or cid not in cluster_scalers: continue
        df_p = monthly_avail[monthly_avail['Nama Produk'] == produk].sort_values('bulan_period')
        df_c = df_p[~df_p['bulan_period'].astype(str).isin(BULAN_EKSKLUDE)]
        if len(df_c) < SEQ_LEN + 4: continue
        med_q = median_qty_produk.get(produk, 1.0)
        qty_v = df_c['qty'].values; n = len(qty_v)
        seq_feats = []
        for step in range(SEQ_LEN):
            idx_cur = n - 1 - (SEQ_LEN - 1 - step)
            if idx_cur < 0: break
            def lag(k): return qty_v[max(idx_cur-k,0)] / max(med_q,1.0)
            r3 = float(np.mean(qty_v[max(0,idx_cur-3):idx_cur])) / max(med_q,1.0) if idx_cur >= 3 else lag(1)
            r6 = float(np.mean(qty_v[max(0,idx_cur-6):idx_cur])) / max(med_q,1.0) if idx_cur >= 6 else r3
            tren = ((qty_v[idx_cur]-qty_v[max(0,idx_cur-3)])/max(abs(qty_v[max(0,idx_cur-3)]),1.0)) if idx_cur>=3 else 0.0
            try:
                curr_bp = df_c['bulan_period'].iloc[idx_cur]
                bulan_step = curr_bp.month; fl_step = KALENDER_LIBUR.get(str(curr_bp), 1.0)
            except Exception:
                bulan_step = bulan_int; fl_step = faktor_libur
            seq_feats.append([lag(1),lag(2),lag(3),lag(6),lag(12),r3,r6,tren,bulan_step,
                              float(le.transform([produk])[0]),fl_step])
        if len(seq_feats) < SEQ_LEN: continue
        seq_per_cluster[cid]['produk'].append(produk)
        seq_per_cluster[cid]['X'].append(np.array(seq_feats[-SEQ_LEN:]))
    hasil = []
    for cid, data in seq_per_cluster.items():
        if not data['produk']: continue
        X_batch = np.array(data['X'])
        n_prod, seq_len_batch, n_feat = X_batch.shape
        X_flat = X_batch.reshape(-1, n_feat)
        X_scaled = cluster_scalers[cid].transform(X_flat).reshape(n_prod, seq_len_batch, n_feat)
        pred_batch = cluster_models[cid].predict(X_scaled, verbose=0).reshape(-1)
        bias_cl = cluster_bias.get(cid, 1.0)
        for produk, pred_log in zip(data['produk'], pred_batch):
            pred_r = float(np.expm1(float(np.clip(pred_log, 0, None))))
            if bias_cl > 0: pred_r = pred_r / bias_cl
            med_q = median_qty_produk.get(produk, 1.0)
            # [v6] CAP_FACTOR_LSTM sekarang 1.0 (sebelumnya 1.5)
            pred_qty = min(max(0.0,max(0.0,pred_r*med_q)*faktor_libur), max_qty_produk.get(produk,pred_r*med_q)*CAP_FACTOR_LSTM)
            hasil.append({'nama_produk':produk,'pred_qty_lstm':round(pred_qty,2),
                          'pred_rev_lstm':round(pred_qty*harga_rata2.get(produk,0),0)})
    return pd.DataFrame(hasil) if hasil else pd.DataFrame(columns=['nama_produk','pred_qty_lstm','pred_rev_lstm'])

def hitung_precision_toleransi(semua_hasil, kolom_pred, toleransi_persen, bulan_test):
    hasil_per_bulan = []
    for i, bulan_pred in enumerate(bulan_test):
        df_b = semua_hasil[i]
        df_valid = df_b[df_b['aktual_qty'] > 0].copy()
        if len(df_valid) == 0: continue
        selisih = (df_valid[kolom_pred] - df_valid['aktual_qty']).abs() / df_valid['aktual_qty'] * 100
        n_akurat = int((selisih <= toleransi_persen).sum())
        n_total = len(df_valid)
        hasil_per_bulan.append({'bulan': str(bulan_pred), 'n_akurat': n_akurat, 'n_total': n_total,
                                 'persen_akurat': n_akurat / n_total * 100})
    return pd.DataFrame(hasil_per_bulan)

print("Fungsi helper siap (v6).")


Fungsi helper siap (v6).


## Sel 4 -- Load, Bersihkan, Agregasi Data

Tahap ini **tidak berubah dari v5** -- CAP_FACTOR hanya memengaruhi tahap prediksi
(Sel 7 dst.), bukan tahap preprocessing. Angka BEFORE/AFTER pada sel ini akan identik
dengan v5 selama file `dataset_toko.csv` sama.

In [3]:
df_raw = pd.read_csv(csv_path, on_bad_lines='skip')
print(f"[BEFORE] Jumlah baris CSV mentah (hasil ekspor Tokopedia Seller): {len(df_raw):,}")
print(f"[BEFORE] Jumlah kolom: {df_raw.shape[1]}")
print(f"[BEFORE] Kolom yang tersedia: {list(df_raw.columns)}")
print()
print("[BEFORE] Contoh 5 baris pertama data mentah:")
display(df_raw.head())
print()
print("[BEFORE] Distribusi Status Terakhir (sebelum difilter):")
display(df_raw['Status Terakhir'].value_counts())


[BEFORE] Jumlah baris CSV mentah (hasil ekspor Tokopedia Seller): 32,689
[BEFORE] Jumlah kolom: 7
[BEFORE] Kolom yang tersedia: ['Invoice', 'Tanggal Pembayaran', 'Status Terakhir', 'Nama Produk', 'Jumlah Produk Dibeli', 'Harga Jual (IDR)', 'Total Penjualan (IDR)']

[BEFORE] Contoh 5 baris pertama data mentah:


,Invoice,Tanggal Pembayaran,Status Terakhir,Nama Produk,Jumlah Produk Dibeli,Harga Jual (IDR),Total Penjualan (IDR)
0,INV/20200507/XX/V/537956873,5/8/2020 10:19,Pesanan Selesai,"AS DINAMO KIPAS ANGIN MODEL COSMOS,MIYAKO,UMUM...",10.0,7000.0,70000.0
1,INV/20200508/XX/V/538745415,5/8/2020 18:00,Pesanan Selesai,"AS DINAMO KIPAS ANGIN MODEL COSMOS,MIYAKO,UMUM...",30.0,7000.0,210000.0
2,INV/20200510/XX/V/540334166,5/10/2020 20:01,Pesanan Selesai,bushing boshing bos bearing kipas angin Rrt/umum,10.0,2200.0,152000.0
3,INV/20200510/XX/V/540334166,5/10/2020 20:01,Pesanan Selesai,AS Exsos kipas lobang 2 model Maspion Nasional.,10.0,6000.0,152000.0
4,INV/20200510/XX/V/540334166,5/10/2020 20:01,Pesanan Selesai,"AS DINAMO KIPAS ANGIN MODEL COSMOS,MIYAKO,UMUM...",10.0,7000.0,152000.0



[BEFORE] Distribusi Status Terakhir (sebelum difilter):


Status Terakhir
Pesanan Selesai       31880
Dibatalkan Pembeli      305
Dibatalkan Penjual      220
Dibatalkan Sistem       165
Sedang Dikirim            1
Name: count, dtype: int64

In [4]:
df = df_raw.copy()
n_sebelum_parsing = len(df)
df['Tanggal Pembayaran'] = pd.to_datetime(df['Tanggal Pembayaran'], format='mixed', errors='coerce')
n_tanggal_invalid = df['Tanggal Pembayaran'].isna().sum()
df = df.dropna(subset=['Tanggal Pembayaran'])
n_sesudah_parsing = len(df)

print(f"[BEFORE] Baris sebelum parsing tanggal : {n_sebelum_parsing:,}")
print(f"[AFTER ] Baris dengan tanggal tidak valid (dibuang) : {n_tanggal_invalid:,}")
print(f"[AFTER ] Baris sesudah parsing tanggal : {n_sesudah_parsing:,}")
print(f"[AFTER ] Selisih : -{n_sebelum_parsing - n_sesudah_parsing:,} baris")
print()
print("[AFTER] Tipe data kolom 'Tanggal Pembayaran' sekarang:", df['Tanggal Pembayaran'].dtype)
print("[AFTER] Contoh isi kolom tanggal setelah parsing:")
display(df[['Tanggal Pembayaran']].head())


[BEFORE] Baris sebelum parsing tanggal : 32,689
[AFTER ] Baris dengan tanggal tidak valid (dibuang) : 114
[AFTER ] Baris sesudah parsing tanggal : 32,575
[AFTER ] Selisih : -114 baris

[AFTER] Tipe data kolom 'Tanggal Pembayaran' sekarang: datetime64[ns]
[AFTER] Contoh isi kolom tanggal setelah parsing:


,Tanggal Pembayaran
0,2020-05-08 10:19:00
1,2020-05-08 18:00:00
2,2020-05-10 20:01:00
3,2020-05-10 20:01:00
4,2020-05-10 20:01:00


In [5]:
n_sebelum_filter_status = len(df)
distribusi_sebelum = df['Status Terakhir'].value_counts()
df = df[df['Status Terakhir'] == 'Pesanan Selesai'].copy()
n_sesudah_filter_status = len(df)

for col in ['Harga Jual (IDR)','Jumlah Produk Dibeli']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# item_revenue dihitung ULANG (bukan memakai kolom 'Total Penjualan (IDR)' bawaan),
# karena kolom bawaan merepresentasikan total 1 invoice, bukan total per item produk.
df['item_revenue'] = (df['Harga Jual (IDR)'] * df['Jumlah Produk Dibeli']).clip(lower=0)
df['bulan_period'] = df['Tanggal Pembayaran'].dt.to_period('M')
df['bulan']        = df['Tanggal Pembayaran'].dt.month

print(f"[BEFORE] Baris sebelum filter status : {n_sebelum_filter_status:,}")
print(f"[BEFORE] Distribusi status transaksi sebelum difilter:")
display(distribusi_sebelum)
print()
print(f"[AFTER ] Baris berstatus 'Pesanan Selesai' (dipertahankan) : {n_sesudah_filter_status:,}")
print(f"[AFTER ] Baris berstatus lain (dibuang) : {n_sebelum_filter_status - n_sesudah_filter_status:,}")
print()
kolom_cek = ['Nama Produk','Harga Jual (IDR)','Jumlah Produk Dibeli']
if 'Total Penjualan (IDR)' in df.columns:
    kolom_cek.append('Total Penjualan (IDR)')
kolom_cek.append('item_revenue')
display(df[kolom_cek].head(5))


[BEFORE] Baris sebelum filter status : 32,575
[BEFORE] Distribusi status transaksi sebelum difilter:


Status Terakhir
Pesanan Selesai       31880
Dibatalkan Pembeli      305
Dibatalkan Penjual      220
Dibatalkan Sistem       165
Sedang Dikirim            1
Name: count, dtype: int64


[AFTER ] Baris berstatus 'Pesanan Selesai' (dipertahankan) : 31,880
[AFTER ] Baris berstatus lain (dibuang) : 695



,Nama Produk,Harga Jual (IDR),Jumlah Produk Dibeli,Total Penjualan (IDR),item_revenue
0,"AS DINAMO KIPAS ANGIN MODEL COSMOS,MIYAKO,UMUM...",7000.0,10.0,70000.0,70000.0
1,"AS DINAMO KIPAS ANGIN MODEL COSMOS,MIYAKO,UMUM...",7000.0,30.0,210000.0,210000.0
2,bushing boshing bos bearing kipas angin Rrt/umum,2200.0,10.0,152000.0,22000.0
3,AS Exsos kipas lobang 2 model Maspion Nasional.,6000.0,10.0,152000.0,60000.0
4,"AS DINAMO KIPAS ANGIN MODEL COSMOS,MIYAKO,UMUM...",7000.0,10.0,152000.0,70000.0


In [6]:
harga_rata2 = df.groupby('Nama Produk')['Harga Jual (IDR)'].mean().to_dict()
bulan_list  = sorted(df['bulan_period'].unique())
split_idx   = int(len(bulan_list) * SPLIT_PCT)
bulan_train = bulan_list[:split_idx]
bulan_test  = bulan_list[split_idx:]

n_transaksi_sebelum_agregasi = len(df)
n_produk_unik_sebelum = df['Nama Produk'].nunique()

produk_contoh = df['Nama Produk'].value_counts().index[0]
contoh_sebelum_agregasi = df[df['Nama Produk']==produk_contoh][
    ['Tanggal Pembayaran','bulan_period','Nama Produk','Jumlah Produk Dibeli','item_revenue']
].sort_values('Tanggal Pembayaran').head(5)

monthly_all = (df.groupby(['bulan_period','bulan','Nama Produk'])
                 .agg(qty=('Jumlah Produk Dibeli','sum'), revenue=('item_revenue','sum'))
                 .reset_index().sort_values(['Nama Produk','bulan_period']))

monthly_train = monthly_all[monthly_all['bulan_period'].isin(bulan_train)]
monthly_test  = monthly_all[monthly_all['bulan_period'].isin(bulan_test)]

print(f"[BEFORE] Jumlah baris transaksi (granularitas harian/per transaksi) : {n_transaksi_sebelum_agregasi:,}")
print(f"[BEFORE] Jumlah produk unik pada seluruh dataset : {n_produk_unik_sebelum}")
print()
print(f"[AFTER ] Jumlah baris monthly_all (granularitas: 1 baris = 1 bulan x 1 produk) : {len(monthly_all):,}")
print(f"[AFTER ] Total bulan aktif dalam rentang data : {len(bulan_list)} bulan")
print(f"[AFTER ]   -> Bulan Training : {len(bulan_train)} bulan ({bulan_train[0]} s.d. {bulan_train[-1]})")
print(f"[AFTER ]   -> Bulan Testing  : {len(bulan_test)} bulan ({bulan_test[0]} s.d. {bulan_test[-1]})")
print()
print(f"[AFTER ] Baris monthly_train : {len(monthly_train):,} | Baris monthly_test : {len(monthly_test):,}")
print()
print(f"[CONTOH] Sebelum agregasi -- transaksi harian produk '{produk_contoh[:50]}...':")
display(contoh_sebelum_agregasi)
print()
print(f"[CONTOH] Sesudah agregasi -- baris bulanan produk yang sama:")
display(monthly_all[monthly_all['Nama Produk']==produk_contoh].head(5))


[BEFORE] Jumlah baris transaksi (granularitas harian/per transaksi) : 31,880
[BEFORE] Jumlah produk unik pada seluruh dataset : 345

[AFTER ] Jumlah baris monthly_all (granularitas: 1 baris = 1 bulan x 1 produk) : 8,224
[AFTER ] Total bulan aktif dalam rentang data : 59 bulan
[AFTER ]   -> Bulan Training : 47 bulan (2020-05 s.d. 2024-03)
[AFTER ]   -> Bulan Testing  : 12 bulan (2024-04 s.d. 2025-03)

[AFTER ] Baris monthly_train : 6,770 | Baris monthly_test : 1,454

[CONTOH] Sebelum agregasi -- transaksi harian produk 'KAPASITOR 1,5UF, 2UF, 2,5UF, 3UF, 3,5UF MIKRO CAPA...':


,Tanggal Pembayaran,bulan_period,Nama Produk,Jumlah Produk Dibeli,item_revenue
1767,2021-01-28 13:16:00,2021-01,"KAPASITOR 1,5UF, 2UF, 2,5UF, 3UF, 3,5UF MIKRO ...",2.0,9000.0
2266,2021-03-21 09:01:00,2021-03,"KAPASITOR 1,5UF, 2UF, 2,5UF, 3UF, 3,5UF MIKRO ...",2.0,9000.0
3391,2021-06-04 14:42:00,2021-06,"KAPASITOR 1,5UF, 2UF, 2,5UF, 3UF, 3,5UF MIKRO ...",4.0,18000.0
3467,2021-06-07 23:10:00,2021-06,"KAPASITOR 1,5UF, 2UF, 2,5UF, 3UF, 3,5UF MIKRO ...",2.0,9000.0
3481,2021-06-08 15:43:00,2021-06,"KAPASITOR 1,5UF, 2UF, 2,5UF, 3UF, 3,5UF MIKRO ...",2.0,9000.0



[CONTOH] Sesudah agregasi -- baris bulanan produk yang sama:


,bulan_period,bulan,Nama Produk,qty,revenue
516,2021-01,1,"KAPASITOR 1,5UF, 2UF, 2,5UF, 3UF, 3,5UF MIKRO ...",2.0,9000.0
710,2021-03,3,"KAPASITOR 1,5UF, 2UF, 2,5UF, 3UF, 3,5UF MIKRO ...",2.0,9000.0
1081,2021-06,6,"KAPASITOR 1,5UF, 2UF, 2,5UF, 3UF, 3,5UF MIKRO ...",15.0,67500.0
1238,2021-07,7,"KAPASITOR 1,5UF, 2UF, 2,5UF, 3UF, 3,5UF MIKRO ...",17.0,76500.0
1397,2021-08,8,"KAPASITOR 1,5UF, 2UF, 2,5UF, 3UF, 3,5UF MIKRO ...",33.0,148500.0


## Sel 5 -- IQR Outlier Clipping (Leak-Free)

Tidak berubah dari v5.

In [7]:
monthly_all['qty_sebelum_iqr'] = monthly_all['qty'].copy()
n_baris_sebelum_iqr = len(monthly_all)
statistik_qty_sebelum = monthly_all['qty'].describe()

iqr_bounds = {}
for p in monthly_train['Nama Produk'].unique():
    vals = monthly_train[monthly_train['Nama Produk']==p]['qty']
    if len(vals) < 4: continue
    Q1, Q3 = vals.quantile(0.25), vals.quantile(0.75)
    IQR = Q3 - Q1
    iqr_bounds[p] = (max(0.0, Q1-IQR_MULTIPLIER*IQR), Q3+IQR_MULTIPLIER*IQR)

def _clip_qty(row):
    b = iqr_bounds.get(row['Nama Produk'])
    return row['qty'] if b is None else float(np.clip(row['qty'], b[0], b[1]))

monthly_all['qty'] = monthly_all.apply(_clip_qty, axis=1)
monthly_train = monthly_all[monthly_all['bulan_period'].isin(bulan_train)]
monthly_test  = monthly_all[monthly_all['bulan_period'].isin(bulan_test)]

n_baris_terdampak = int((monthly_all['qty'] != monthly_all['qty_sebelum_iqr']).sum())
print(f"[BEFORE] Jumlah baris monthly_all sebelum IQR clipping : {n_baris_sebelum_iqr:,}")
print(f"[AFTER ] Jumlah produk dengan batas IQR berhasil dihitung : {len(iqr_bounds)}")
print(f"[AFTER ] Jumlah baris yang nilai qty-nya berubah (outlier di-clip) : {n_baris_terdampak:,} dari {n_baris_sebelum_iqr:,} baris")
print()
print("[BEFORE] Statistik deskriptif qty SEBELUM clipping:")
display(statistik_qty_sebelum)
print("[AFTER ] Statistik deskriptif qty SESUDAH clipping:")
display(monthly_all['qty'].describe())

monthly_all = monthly_all.drop(columns=['qty_sebelum_iqr'])


[BEFORE] Jumlah baris monthly_all sebelum IQR clipping : 8,224
[AFTER ] Jumlah produk dengan batas IQR berhasil dihitung : 292
[AFTER ] Jumlah baris yang nilai qty-nya berubah (outlier di-clip) : 526 dari 8,224 baris

[BEFORE] Statistik deskriptif qty SEBELUM clipping:


count    8224.000000
mean       23.538059
std        73.581598
min         1.000000
25%         3.000000
50%         8.000000
75%        22.000000
max      3312.000000
Name: qty, dtype: float64

[AFTER ] Statistik deskriptif qty SESUDAH clipping:


count    8224.000000
mean       21.319811
std        45.591878
min         1.000000
25%         3.000000
50%         8.000000
75%        21.000000
max       904.500000
Name: qty, dtype: float64

## Sel 6 -- Filter Produk Layak Pemodelan (Funnel)

Tidak berubah dari v5.

In [8]:
n_produk_unik_training_awal = monthly_train['Nama Produk'].nunique()
produk_count = monthly_train.groupby('Nama Produk')['bulan_period'].count()
produk_layak_awal = produk_count[produk_count >= MIN_BULAN].index.tolist()
n_lolos_min_bulan = len(produk_layak_awal)
n_gugur_min_bulan = n_produk_unik_training_awal - n_lolos_min_bulan

bulan_train_bersih = [b for b in bulan_train if str(b) not in BULAN_ANOMALI]
bulan_cek_aktif = bulan_train_bersih[-MIN_BULAN_AKTIF:]

def cek_aktif(p):
    df_p = monthly_train[monthly_train['Nama Produk']==p]
    return df_p[df_p['bulan_period'].isin(bulan_cek_aktif)]['qty'].sum() > 0

produk_layak = [p for p in produk_layak_awal if cek_aktif(p)]
n_lolos_aktif = len(produk_layak)
n_gugur_aktif = n_lolos_min_bulan - n_lolos_aktif
produk_arima_l = list(produk_layak)

mean_qty_produk, median_qty_produk, max_qty_produk = {}, {}, {}
for p in produk_layak:
    vals = monthly_train[(monthly_train['Nama Produk']==p) &
                          (~monthly_train['bulan_period'].astype(str).isin(BULAN_ANOMALI))]['qty'].values
    mean_qty_produk[p]   = max(float(vals.mean()), 1.0) if len(vals)>0 else 1.0
    median_qty_produk[p] = max(float(np.median(vals)), 1.0) if len(vals)>0 else 1.0
    max_qty_produk[p]    = max(float(vals.max()), 1.0) if len(vals)>0 else 1.0

funnel = pd.DataFrame({
    'Tahap Filter': [
        'Total produk unik (partisi training)',
        f'Riwayat >= {MIN_BULAN} bulan (MIN_BULAN)',
        f'Aktif pada {MIN_BULAN_AKTIF} bulan terakhir training (MIN_BULAN_AKTIF)',
        'Dilatih ARIMA maupun LSTM (populasi identik)'
    ],
    'Jumlah Produk': [n_produk_unik_training_awal, n_lolos_min_bulan, n_lolos_aktif, len(produk_arima_l)],
    'Produk Gugur pada Tahap Ini': ['-', n_gugur_min_bulan, n_gugur_aktif, 0]
})
print("[FUNNEL] Corong filter kelayakan produk:")
display(funnel)
print(f"[AFTER ] Produk layak dimodelkan (populasi ARIMA = populasi LSTM) : {len(produk_layak)}")
print(f"         -> Rasio produk layak terhadap total training : {len(produk_layak)/n_produk_unik_training_awal*100:.1f}%")


[FUNNEL] Corong filter kelayakan produk:


,Tahap Filter,Jumlah Produk,Produk Gugur pada Tahap Ini
0,Total produk unik (partisi training),325,-
1,Riwayat >= 15 bulan (MIN_BULAN),192,133
2,Aktif pada 3 bulan terakhir training (MIN_BULA...,166,26
3,Dilatih ARIMA maupun LSTM (populasi identik),166,0


[AFTER ] Produk layak dimodelkan (populasi ARIMA = populasi LSTM) : 166
         -> Rasio produk layak terhadap total training : 51.1%


## Sel 7 -- Training ARIMA Paralel

Tidak berubah -- `CAP_FACTOR_ARIMA` baru dipakai nanti pada tahap **prediksi**
(Sel 10, 12, 17), bukan pada tahap fitting order di sel ini.

In [9]:
print(f"[BEFORE] Jumlah produk yang AKAN dilatih ARIMA : {len(produk_arima_l)}")
print(f"[BEFORE] Status model : belum ada order ARIMA yang dipilih untuk produk manapun.")
print()

t0 = time.time()
tracemalloc.start()
best_orders_arima, bias_correction = latih_semua_arima(produk_arima_l, monthly_train)
t_arima = time.time() - t0
mem_arima = tracemalloc.get_traced_memory()[1] / 1024**2
tracemalloc.stop()

n_berhasil = len(best_orders_arima)
n_gagal = len(produk_arima_l) - n_berhasil
print(f"[AFTER ] Produk berhasil dilatih (order ARIMA terpilih) : {n_berhasil} dari {len(produk_arima_l)}")
print(f"[AFTER ] Produk gagal dilatih : {n_gagal}")
print(f"[AFTER ] Waktu training : {t_arima:.1f} detik | Memori puncak : {mem_arima:.1f} MB | n_jobs={N_JOBS}")

order_count = Counter(best_orders_arima.values())
if order_count:
    order_top, freq_top = order_count.most_common(1)[0]
    print(f"[AFTER ] Order ARIMA terpopuler: {order_top} ({freq_top}/{n_berhasil} produk = {freq_top/n_berhasil*100:.1f}%)")
if bias_correction:
    print(f"[AFTER ] Rentang bias_correction: [{min(bias_correction.values()):.2f}, {max(bias_correction.values()):.2f}]")

print()
print("[AFTER] Distribusi 5 order ARIMA terpopuler:")
display(pd.DataFrame(order_count.most_common(5), columns=['Order (p,d,q)', 'Jumlah Produk']))


[BEFORE] Jumlah produk yang AKAN dilatih ARIMA : 166
[BEFORE] Status model : belum ada order ARIMA yang dipilih untuk produk manapun.

[AFTER ] Produk berhasil dilatih (order ARIMA terpilih) : 166 dari 166
[AFTER ] Produk gagal dilatih : 0
[AFTER ] Waktu training : 50.6 detik | Memori puncak : 1.1 MB | n_jobs=-1
[AFTER ] Order ARIMA terpopuler: (0, 1, 1) (107/166 produk = 64.5%)
[AFTER ] Rentang bias_correction: [0.70, 1.50]

[AFTER] Distribusi 5 order ARIMA terpopuler:


,"Order (p,d,q)",Jumlah Produk
0,"(0, 1, 1)",107
1,"(0, 1, 2)",12
2,"(0, 1, 3)",12
3,"(2, 1, 0)",7
4,"(2, 1, 2)",6


## Sel 8 -- Rekayasa Fitur dan Klasterisasi LSTM

Tidak berubah dari v5.

In [10]:
sorted_prods   = sorted(produk_layak, key=lambda p: median_qty_produk[p])
cluster_size   = max(1, len(sorted_prods)//N_CLUSTER)
produk_cluster = {p: min(i//cluster_size, N_CLUSTER-1) for i,p in enumerate(sorted_prods)}

le  = LabelEncoder()
mtr = monthly_train[monthly_train['Nama Produk'].isin(produk_layak)].copy()
n_baris_sebelum_fitur = len(mtr)

mtr['produk_id']  = le.fit_transform(mtr['Nama Produk'])
mtr['median_qty'] = mtr['Nama Produk'].map(median_qty_produk)
mtr['cluster_id'] = mtr['Nama Produk'].map(produk_cluster)

for lag in [1,2,3,6,12]:
    mtr[f'lag{lag}_r'] = mtr.groupby('Nama Produk', sort=True)['qty'].transform(lambda x: x.shift(lag)) / mtr['median_qty'].clip(lower=1.0)
for w, col in [(3,'roll3_r'),(6,'roll6_r')]:
    mtr[col] = mtr.groupby('Nama Produk', sort=True)['qty'].transform(lambda x: x.shift(1).rolling(w, min_periods=1).mean()) / mtr['median_qty'].clip(lower=1.0)
mtr['tren_3m'] = mtr.groupby('Nama Produk', sort=True)['qty'].transform(
    lambda x: x.shift(1).rolling(3, min_periods=2).apply(lambda v: (v[-1]-v[0])/max(abs(v[0]),1), raw=True))
mtr['faktor_libur'] = mtr['bulan_period'].astype(str).map(lambda x: KALENDER_LIBUR.get(x, 1.0))
mtr['log_rasio']    = np.log1p((mtr['qty']/mtr['median_qty'].clip(lower=1.0)).clip(0, 8.0))

n_baris_sebelum_dropna = len(mtr)
n_nan = int(mtr[FEATURES].isna().any(axis=1).sum())
mtr_clean = mtr.dropna()
mtr_clean = mtr_clean[~mtr_clean['bulan_period'].astype(str).isin(BULAN_EKSKLUDE)].copy()
mtr_clean = mtr_clean.sort_values(['cluster_id','Nama Produk','bulan_period']).reset_index(drop=True)

print(f"[BEFORE] Jumlah baris (166 produk layak, belum ada fitur) : {n_baris_sebelum_fitur:,}")
print(f"[AFTER ] Baris mtr_clean sesudah dropna() + eksklusi bulan anomali : {len(mtr_clean):,}")
print(f"[AFTER ] Total penyusutan : {n_baris_sebelum_fitur:,} -> {len(mtr_clean):,} "
      f"(-{n_baris_sebelum_fitur - len(mtr_clean):,} baris, "
      f"{((n_baris_sebelum_fitur-len(mtr_clean))/n_baris_sebelum_fitur*100):.1f}%)")
print()
print("[AFTER] Distribusi jumlah sampel per klaster:")
display(mtr_clean.groupby('cluster_id').size().rename('jumlah_sampel').reset_index())


[BEFORE] Jumlah baris (166 produk layak, belum ada fitur) : 5,193
[AFTER ] Baris mtr_clean sesudah dropna() + eksklusi bulan anomali : 3,201
[AFTER ] Total penyusutan : 5,193 -> 3,201 (-1,992 baris, 38.4%)

[AFTER] Distribusi jumlah sampel per klaster:


,cluster_id,jumlah_sampel
0,0,377
1,1,521
2,2,685
3,3,787
4,4,831


## Sel 9 -- Training LSTM per Klaster

Tidak berubah -- `CAP_FACTOR_LSTM` baru dipakai pada tahap **prediksi**
(Sel 10, 12, 17), bukan pada tahap training bobot model di sel ini.

In [11]:
t0 = time.time()
tracemalloc.start()
cluster_models, cluster_scalers, cluster_bias = {}, {}, {}
log_klaster = []

for cid in range(N_CLUSTER):
    df_cl = mtr_clean[mtr_clean['cluster_id']==cid].reset_index(drop=True)
    n_baris_klaster = len(df_cl)
    if len(df_cl) < 40:
        log_klaster.append({'cluster_id': cid, 'baris_mtr_clean': n_baris_klaster,
                             'sampel_sequence_3d': 0, 'epoch': 0, 'status': 'DILEWATI (<40 baris)'})
        continue
    set_seed_ulang(cid)
    scaler = MinMaxScaler()
    X_sc = scaler.fit_transform(df_cl[FEATURES].values.astype(float))
    y_all = df_cl['log_rasio'].values
    X_seq, y_seq = [], []
    for _, grp in df_cl.groupby('Nama Produk', sort=True):
        local_idx = list(grp.sort_values('bulan_period').index)
        for i in range(len(local_idx)-SEQ_LEN):
            X_seq.append(X_sc[local_idx[i:i+SEQ_LEN]])
            y_seq.append(y_all[local_idx[i+SEQ_LEN]])
    if len(X_seq) < 20:
        log_klaster.append({'cluster_id': cid, 'baris_mtr_clean': n_baris_klaster,
                             'sampel_sequence_3d': len(X_seq), 'epoch': 0, 'status': 'DILEWATI (<20 sequence)'})
        continue
    X_3d, y_arr = np.array(X_seq), np.array(y_seq)
    model = Sequential([LSTM(64, input_shape=(SEQ_LEN,len(FEATURES)), return_sequences=True),
                        BatchNormalization(), Dropout(0.15),
                        LSTM(32, return_sequences=False),
                        BatchNormalization(), Dropout(0.15),
                        Dense(16, activation='relu'), Dense(1)])
    model.compile(loss=tf.keras.losses.Huber(delta=1.0), optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4))
    hist = model.fit(X_3d, y_arr, epochs=500, batch_size=16, validation_split=0.2, shuffle=False,
        callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=40, restore_best_weights=True),
                   tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=15, factor=0.5, min_lr=1e-6, verbose=0)],
        verbose=0)
    cluster_models[cid]=model; cluster_scalers[cid]=scaler; cluster_bias[cid]=1.0
    log_klaster.append({'cluster_id': cid, 'baris_mtr_clean': n_baris_klaster,
                         'sampel_sequence_3d': len(X_seq), 'epoch': len(hist.history['loss']), 'status': 'BERHASIL'})
    print(f"  Klaster {cid} selesai: {n_baris_klaster} baris 2D -> {len(X_seq)} sequence 3D | epoch={len(hist.history['loss'])}")

t_lstm = time.time() - t0
mem_lstm = tracemalloc.get_traced_memory()[1] / 1024**2
tracemalloc.stop()

df_log_klaster = pd.DataFrame(log_klaster)
print("[AFTER] Rangkuman training LSTM per klaster:")
display(df_log_klaster)
print(f"[AFTER ] Total klaster berhasil dilatih : {len(cluster_models)} dari {N_CLUSTER}")
print(f"[AFTER ] Total waktu training LSTM : {t_lstm:.1f} detik | Memori puncak : {mem_lstm:.1f} MB")


  Klaster 0 selesai: 377 baris 2D -> 190 sequence 3D | epoch=73
  Klaster 1 selesai: 521 baris 2D -> 332 sequence 3D | epoch=69
  Klaster 2 selesai: 685 baris 2D -> 487 sequence 3D | epoch=53
  Klaster 3 selesai: 787 baris 2D -> 589 sequence 3D | epoch=59
  Klaster 4 selesai: 831 baris 2D -> 629 sequence 3D | epoch=55
[AFTER] Rangkuman training LSTM per klaster:


,cluster_id,baris_mtr_clean,sampel_sequence_3d,epoch,status
0,0,377,190,73,BERHASIL
1,1,521,332,69,BERHASIL
2,2,685,487,53,BERHASIL
3,3,787,589,59,BERHASIL
4,4,831,629,55,BERHASIL


[AFTER ] Total klaster berhasil dilatih : 5 dari 5
[AFTER ] Total waktu training LSTM : 190.1 detik | Memori puncak : 47.7 MB


## Sel 10 -- Evaluasi Rolling 12 Bulan

Sel ini adalah tempat **CAP_FACTOR_ARIMA (0.8)** dan **CAP_FACTOR_LSTM (1.0)** yang
baru mulai memengaruhi angka hasil, karena `prediksi_arima()` dan `prediksi_lstm()`
dipanggil di sini dan keduanya membaca kedua konstanta tersebut dari Sel 0.

In [12]:
print(f"[BEFORE] Jumlah bulan yang akan dievaluasi (walk-forward) : {len(bulan_test)}")
print(f"[BEFORE] Jumlah produk yang diprediksi setiap bulan : {len(produk_layak)}")
print()

t0 = time.time()
semua_hasil, log_bulan = [], []
for bulan_pred in bulan_test:
    avail = monthly_all[monthly_all['bulan_period'] < bulan_pred]
    df_a = prediksi_arima(bulan_pred, avail, produk_layak, best_orders_arima, bias_correction, harga_rata2, max_qty_produk)
    df_l = prediksi_lstm(bulan_pred, avail, produk_layak, cluster_models, cluster_scalers, cluster_bias,
                          produk_cluster, le, harga_rata2, median_qty_produk, max_qty_produk)
    df_m = pd.merge(df_a, df_l, on='nama_produk', how='outer').fillna(0)
    aktual_b = monthly_test[monthly_test['bulan_period']==bulan_pred][['Nama Produk','qty','revenue']].rename(
        columns={'Nama Produk':'nama_produk','qty':'aktual_qty','revenue':'aktual_rev'})
    df_m = pd.merge(df_m, aktual_b, on='nama_produk', how='left').fillna(0)
    semua_hasil.append(df_m)
    log_bulan.append({'bulan': str(bulan_pred), 'n_produk_diprediksi': len(df_m),
                       'aktual': df_m['aktual_rev'].sum(),
                       'pred_arima': df_m['pred_rev_arima'].sum(), 'pred_lstm': df_m['pred_rev_lstm'].sum()})
df_log = pd.DataFrame(log_bulan)
t_eval = time.time() - t0

print(f"[AFTER ] Evaluasi rolling 12 bulan selesai dalam {t_eval:.1f} detik.")
print("[AFTER] Ringkasan revenue aktual vs prediksi (v6, CAP_FACTOR_ARIMA=0.8, CAP_FACTOR_LSTM=1.0):")
display(df_log[['bulan','aktual','pred_arima','pred_lstm']])


[BEFORE] Jumlah bulan yang akan dievaluasi (walk-forward) : 12
[BEFORE] Jumlah produk yang diprediksi setiap bulan : 166

[AFTER ] Evaluasi rolling 12 bulan selesai dalam 29.8 detik.
[AFTER] Ringkasan revenue aktual vs prediksi (v6, CAP_FACTOR_ARIMA=0.8, CAP_FACTOR_LSTM=1.0):


,bulan,aktual,pred_arima,pred_lstm
0,2024-04,12248900.0,13201450.0,9597641.0
1,2024-05,6114200.0,11469706.0,8263169.0
2,2024-06,1745800.0,2105448.0,1554664.0
3,2024-07,3420200.0,4912725.0,3627548.0
4,2024-08,5895225.0,9592328.0,7125549.0
5,2024-09,12046400.0,17213791.0,14255807.0
6,2024-10,14812400.0,16944532.0,14893957.0
7,2024-11,14015450.0,16258296.0,15449815.0
8,2024-12,4747175.0,6094097.0,6104674.0
9,2025-01,7832075.0,8921324.0,8820682.0


## Sel 11 -- Metrik Evaluasi Akhir: MAE dan RMSE (v6)

Hasil MAE ARIMA di sel ini seharusnya mendekati **Rp2.408.274** (bukan lagi
Rp2.467.634 seperti v5), sesuai temuan eksperimen CAP_FACTOR_ARIMA. MAE LSTM
seharusnya tetap **Rp1.327.530**, karena eksperimen CAP_FACTOR_LSTM menunjukkan
tidak ada perubahan nilai pada rentang 1.0--2.5.

In [13]:
mae_agg_a  = float(mean_absolute_error(df_log['aktual'], df_log['pred_arima']))
rmse_agg_a = float(np.sqrt(mean_squared_error(df_log['aktual'], df_log['pred_arima'])))
mae_agg_l  = float(mean_absolute_error(df_log['aktual'], df_log['pred_lstm']))
rmse_agg_l = float(np.sqrt(mean_squared_error(df_log['aktual'], df_log['pred_lstm'])))

hasil_akhir = pd.DataFrame({
    'Model':  ['ARIMA', 'LSTM'],
    'MAE (Rp)':  [f"{mae_agg_a:,.0f}", f"{mae_agg_l:,.0f}"],
    'RMSE (Rp)': [f"{rmse_agg_a:,.0f}", f"{rmse_agg_l:,.0f}"],
    'Waktu Training (detik)': [round(t_arima,1), round(t_lstm,1)],
})
print("[AFTER v6] MAE/RMSE final dengan CAP_FACTOR_ARIMA=0.8 & CAP_FACTOR_LSTM=1.0:")
display(hasil_akhir)
print(f"Selisih MAE (ARIMA - LSTM) : Rp{mae_agg_a - mae_agg_l:,.0f}  "
      f"({(mae_agg_a - mae_agg_l)/mae_agg_a*100:.1f}% lebih rendah pada LSTM)")
print()
print("[BANDINGKAN] MAE ARIMA v5 (CAP=1.0) = Rp2,467,634  vs  v6 (CAP=0.8) = "
      f"Rp{mae_agg_a:,.0f}  -> selisih {(2467634-mae_agg_a)/2467634*100:.1f}%")


[AFTER v6] MAE/RMSE final dengan CAP_FACTOR_ARIMA=0.8 & CAP_FACTOR_LSTM=1.0:


,Model,MAE (Rp),RMSE (Rp),Waktu Training (detik)
0,ARIMA,"2,408,276","2,870,852",50.6
1,LSTM,"1,327,530","1,587,740",190.1


Selisih MAE (ARIMA - LSTM) : Rp1,080,746  (44.9% lebih rendah pada LSTM)

[BANDINGKAN] MAE ARIMA v5 (CAP=1.0) = Rp2,467,634  vs  v6 (CAP=0.8) = Rp2,408,276  -> selisih 2.4%


## Sel 12 -- Cek Overfitting (Perbandingan MAE Training vs Testing)

Tidak berubah strukturnya dari v5 -- hanya angka MAE ARIMA yang berubah mengikuti
CAP_FACTOR_ARIMA baru.

In [14]:
bulan_train_cek = bulan_train[-len(bulan_test):]
print(f"[BEFORE] Bulan yang dipakai utk cek overfitting: {bulan_train_cek[0]} s.d. {bulan_train_cek[-1]}")

t0 = time.time()
log_train_cek = []
for bulan_pred in bulan_train_cek:
    avail = monthly_all[monthly_all['bulan_period'] < bulan_pred]
    df_a = prediksi_arima(bulan_pred, avail, produk_layak, best_orders_arima, bias_correction, harga_rata2, max_qty_produk)
    df_l = prediksi_lstm(bulan_pred, avail, produk_layak, cluster_models, cluster_scalers, cluster_bias,
                          produk_cluster, le, harga_rata2, median_qty_produk, max_qty_produk)
    df_m = pd.merge(df_a, df_l, on='nama_produk', how='outer').fillna(0)
    aktual_b = monthly_train[monthly_train['bulan_period']==bulan_pred][['Nama Produk','qty','revenue']].rename(
        columns={'Nama Produk':'nama_produk','qty':'aktual_qty','revenue':'aktual_rev'})
    df_m = pd.merge(df_m, aktual_b, on='nama_produk', how='left').fillna(0)
    log_train_cek.append({'aktual': df_m['aktual_rev'].sum(), 'pred_arima': df_m['pred_rev_arima'].sum(),
                           'pred_lstm': df_m['pred_rev_lstm'].sum()})
df_log_train_cek = pd.DataFrame(log_train_cek)

mae_train_a = float(mean_absolute_error(df_log_train_cek['aktual'], df_log_train_cek['pred_arima']))
mae_train_l = float(mean_absolute_error(df_log_train_cek['aktual'], df_log_train_cek['pred_lstm']))

tabel_overfit = pd.DataFrame({
    'Model': ['ARIMA', 'LSTM'],
    'MAE Training (Rp)': [f"{mae_train_a:,.0f}", f"{mae_train_l:,.0f}"],
    'MAE Testing (Rp)':  [f"{mae_agg_a:,.0f}",   f"{mae_agg_l:,.0f}"],
    'Rasio Test/Train':  [f"{mae_agg_a/max(mae_train_a,1):.2f}x", f"{mae_agg_l/max(mae_train_l,1):.2f}x"],
})
print(f"[AFTER ] Cek overfitting selesai dalam {time.time()-t0:.1f} detik.")
display(tabel_overfit)
for nama, mae_tr, mae_te in [('ARIMA', mae_train_a, mae_agg_a), ('LSTM', mae_train_l, mae_agg_l)]:
    rasio = mae_te / max(mae_tr, 1)
    status = 'INDIKASI OVERFITTING (>2x)' if rasio > 2.0 else ('PERLU DIPERHATIKAN (1.5-2x)' if rasio > 1.5 else 'STABIL (tidak overfitting)')
    print(f"  {nama}: {status}")


[BEFORE] Bulan yang dipakai utk cek overfitting: 2023-04 s.d. 2024-03
[AFTER ] Cek overfitting selesai dalam 25.6 detik.


,Model,MAE Training (Rp),MAE Testing (Rp),Rasio Test/Train
0,ARIMA,"4,092,391","2,408,276",0.59x
1,LSTM,"6,911,531","1,327,530",0.19x


  ARIMA: STABIL (tidak overfitting)
  LSTM: STABIL (tidak overfitting)


## Sel 13 -- Precision@10 per Bulan

Tidak berubah strukturnya dari v5. Karena CAP_FACTOR hanya membatasi nilai *ekstrem*
prediksi, Precision@10 kemungkinan besar tidak banyak berubah (batas 0.8x dan 1.0x
jarang aktif kecuali pada produk dengan lonjakan tak wajar).

In [15]:
hasil_precision = []
for i, bulan_pred in enumerate(bulan_test):
    df_m = semua_hasil[i]
    top_aktual = set(df_m.nlargest(TOP_N, 'aktual_rev')['nama_produk'])
    baris = {'bulan': str(bulan_pred), 'n_top_aktual': len(top_aktual)}
    for key, kolom in [('arima', 'pred_rev_arima'), ('lstm', 'pred_rev_lstm')]:
        top_pred = set(df_m.nlargest(TOP_N, kolom)['nama_produk'])
        overlap = len(top_pred & top_aktual)
        baris[f'precision10_{key}'] = overlap / TOP_N
        baris[f'overlap_{key}'] = overlap
    hasil_precision.append(baris)
df_precision = pd.DataFrame(hasil_precision)
print("[AFTER v6] Overlap antara Top-10 aktual dan Top-10 prediksi tiap bulan:")
display(df_precision)
print(f"Precision@10 rata-rata -- ARIMA: {df_precision['precision10_arima'].mean():.3f} | "
      f"LSTM: {df_precision['precision10_lstm'].mean():.3f}")


[AFTER v6] Overlap antara Top-10 aktual dan Top-10 prediksi tiap bulan:


,bulan,n_top_aktual,precision10_arima,overlap_arima,precision10_lstm,overlap_lstm
0,2024-04,10,0.3,3,0.3,3
1,2024-05,10,0.2,2,0.2,2
2,2024-06,10,0.2,2,0.0,0
3,2024-07,10,0.3,3,0.3,3
4,2024-08,10,0.4,4,0.3,3
5,2024-09,10,0.4,4,0.4,4
6,2024-10,10,0.5,5,0.4,4
7,2024-11,10,0.4,4,0.3,3
8,2024-12,10,0.4,4,0.0,0
9,2025-01,10,0.5,5,0.3,3


Precision@10 rata-rata -- ARIMA: 0.358 | LSTM: 0.267


## Sel 14 -- Precision@Toleransi ±20% (Bulan Testing Terakhir)

In [16]:
df_bulan_akhir = semua_hasil[-1]
n_total_produk_bulan_akhir = len(df_bulan_akhir)
df_valid_akhir = df_bulan_akhir[df_bulan_akhir['aktual_qty'] > 0].copy()
n_produk_terjual = len(df_valid_akhir)

print(f"[BEFORE] Total produk yang diprediksi pada bulan {bulan_test[-1]} : {n_total_produk_bulan_akhir}")
print(f"[AFTER ] Produk yang benar-benar TERJUAL pada bulan tsb : {n_produk_terjual}")
print()

hasil_tol_20 = []
for key, kolom in [('ARIMA', 'pred_qty_arima'), ('LSTM', 'pred_qty_lstm')]:
    selisih = (df_valid_akhir[kolom] - df_valid_akhir['aktual_qty']).abs() / df_valid_akhir['aktual_qty'] * 100
    n_akurat = int((selisih <= TOLERANSI_KETAT).sum())
    hasil_tol_20.append({'Model': key, 'Produk Akurat': n_akurat, 'Total Produk': n_produk_terjual,
                          'Persentase': f"{n_akurat/n_produk_terjual*100:.1f}%"})
display(pd.DataFrame(hasil_tol_20))
print(f"Bulan: {bulan_test[-1]} | Toleransi: +-{TOLERANSI_KETAT}%")


[BEFORE] Total produk yang diprediksi pada bulan 2025-03 : 166
[AFTER ] Produk yang benar-benar TERJUAL pada bulan tsb : 107



,Model,Produk Akurat,Total Produk,Persentase
0,ARIMA,19,107,17.8%
1,LSTM,19,107,17.8%


Bulan: 2025-03 | Toleransi: +-20%


## Sel 15 -- Precision@Toleransi ±60% (Agregat 12 Bulan)

In [17]:
hasil_tol_arima = hitung_precision_toleransi(semua_hasil, 'pred_qty_arima', TOLERANSI_LONGGAR, bulan_test)
hasil_tol_lstm  = hitung_precision_toleransi(semua_hasil, 'pred_qty_lstm',  TOLERANSI_LONGGAR, bulan_test)
print("[AFTER v6] Persentase produk akurat (toleransi longgar +-60%) per bulan:")
display(hasil_tol_arima.merge(hasil_tol_lstm, on='bulan', suffixes=('_arima','_lstm')))

ringkasan_tol = pd.DataFrame({
    'Model': ['ARIMA', 'LSTM'],
    f'Rata-rata % Akurat (tol {TOLERANSI_LONGGAR}%)': [f"{hasil_tol_arima['persen_akurat'].mean():.1f}%",
                                                         f"{hasil_tol_lstm['persen_akurat'].mean():.1f}%"],
})
display(ringkasan_tol)


[AFTER v6] Persentase produk akurat (toleransi longgar +-60%) per bulan:


,bulan,n_akurat_arima,n_total_arima,persen_akurat_arima,n_akurat_lstm,n_total_lstm,persen_akurat_lstm
0,2024-04,58,128,45.312500,39,128,30.468750
1,2024-05,28,71,39.436620,25,71,35.211268
2,2024-06,9,31,29.032258,9,31,29.032258
3,2024-07,33,60,55.000000,21,60,35.000000
4,2024-08,34,69,49.275362,26,69,37.681159
5,2024-09,52,117,44.444444,52,117,44.444444
6,2024-10,57,116,49.137931,49,116,42.241379
7,2024-11,51,122,41.803279,50,122,40.983607
8,2024-12,32,72,44.444444,21,72,29.166667
9,2025-01,58,91,63.736264,40,91,43.956044


,Model,Rata-rata % Akurat (tol 60%)
0,ARIMA,46.5%
1,LSTM,38.3%


## Sel 16 -- Tabel Top-10 Aktual vs Prediksi (Format Presentasi)

In [18]:
print(f"[BEFORE] Total baris df_bulan_akhir (semua produk, bulan {bulan_test[-1]}) : {len(df_bulan_akhir)}")

top10_aktual = df_bulan_akhir.nlargest(TOP_N, 'aktual_rev')[['nama_produk','aktual_qty','aktual_rev']].reset_index(drop=True)
top10_pred_arima = df_bulan_akhir.nlargest(TOP_N, 'pred_rev_arima')[['nama_produk','pred_qty_arima','pred_rev_arima']].reset_index(drop=True)
top10_pred_lstm  = df_bulan_akhir.nlargest(TOP_N, 'pred_rev_lstm')[['nama_produk','pred_qty_lstm','pred_rev_lstm']].reset_index(drop=True)
top10_aktual.insert(0, 'Peringkat', range(1, TOP_N+1))
top10_pred_arima.insert(0, 'Peringkat', range(1, TOP_N+1))
top10_pred_lstm.insert(0, 'Peringkat', range(1, TOP_N+1))

print(f"Bulan: {bulan_test[-1]}")
print("Top-10 AKTUAL:")
display(top10_aktual)
print("Top-10 PREDIKSI ARIMA (v6, CAP=0.8):")
display(top10_pred_arima)
print("Top-10 PREDIKSI LSTM (v6, CAP=1.0):")
display(top10_pred_lstm)

overlap_arima = set(top10_aktual['nama_produk']) & set(top10_pred_arima['nama_produk'])
overlap_lstm  = set(top10_aktual['nama_produk']) & set(top10_pred_lstm['nama_produk'])
print(f"[AFTER] Overlap dengan aktual -- ARIMA: {len(overlap_arima)}/{TOP_N} | LSTM: {len(overlap_lstm)}/{TOP_N}")


[BEFORE] Total baris df_bulan_akhir (semua produk, bulan 2025-03) : 166
Bulan: 2025-03
Top-10 AKTUAL:


,Peringkat,nama_produk,aktual_qty,aktual_rev
0,1,KAPASITOR 35 UF 450VAC CAPASITOR BULET CONDENS...,23.0,793500.0
1,2,KAPASITOR POMPA AIR 6 UF 450VAC CAPASITOR BULE...,58.0,690200.0
2,3,IMPELLER KUNINGAN PS 226 230 SHIMIZU POMPA AIR...,6.0,686000.0
3,4,"KAPASITOR 1,5UF, 2UF, 2,5UF, 3UF, 3,5UF MIKRO ...",118.0,531000.0
4,5,KAPASITOR POMPA AIR 25UF 450VAC CAPASITOR BULE...,20.0,518000.0
5,6,KAPASITOR POMPA AIR 20UF 450VAC CAPASITOR BULE...,21.0,459900.0
6,7,"KAPASITOR 1,5UF, 2UF, 2,5UF, 3UF, 3,5UF MIKRO ...",92.0,455400.0
7,8,KAPASITOR POMPA AIR 14 UF 450VAC CAPASITOR BUL...,25.0,447500.0
8,9,carbon arang blender Philips brush karbon kale...,98.0,441000.0
9,10,OTOMATIS PM5 PRESSURE SWITCH JETPUMP POMPA AIR...,8.0,440000.0


Top-10 PREDIKSI ARIMA (v6, CAP=0.8):


,Peringkat,nama_produk,pred_qty_arima,pred_rev_arima
0,1,ELEMEN TUTUP ATAS MAGIC COM JAR PEMANAS TOP HE...,59.27,540829.0
1,2,KAPASITOR 50 UF 50UF 450VAC 450 VAC CAPASITOR ...,10.66,510107.0
2,3,"AS kipas angin model Maspion 20,5 cm.AS Maspio...",71.69,463842.0
3,4,GEAR GIGI MIXER MIYAKO HM 620 625 650 GRIGI SE...,60.36,457021.0
4,5,KAPASITOR 8 UF 450V KOTAK KABEL SAN EI CAPACIT...,32.38,434216.0
5,6,OTOMATIS PM5 PRESSURE SWITCH JETPUMP POMPA AIR...,7.60,429653.0
6,7,GEARBOX KIPAS ANGIN COSMOS / SEKAI / MIYAKO / ...,44.70,371109.0
7,8,KAPASITOR 35 UF 450VAC CAPASITOR BULET CONDENS...,9.42,344137.0
8,9,KAPASITOR 5 UF 450 V KOTAK CAPACITOR PETAK POM...,28.59,279614.0
9,10,SEAL PDL 255 PEDROLO SIL POMPA AIR MS 14 PDR K...,41.64,279516.0


Top-10 PREDIKSI LSTM (v6, CAP=1.0):


,Peringkat,nama_produk,pred_qty_lstm,pred_rev_lstm
0,1,KAPASITOR POMPA AIR 20UF 450VAC CAPASITOR BULE...,24.57,585677.0
1,2,"KAPASITOR 1,5UF, 2UF, 2,5UF, 3UF, 3,5UF MIKRO ...",99.50,476850.0
2,3,PINION PANASONIC NASIONAL PEN STOP TARIKAN GEA...,119.12,424430.0
3,4,SEAL PDL 255 PEDROLO SIL POMPA AIR MS 14 PDR K...,59.20,397392.0
4,5,KAPASITOR POMPA AIR 16UF 450VAC CAPASITOR BULE...,18.38,337249.0
5,6,"AS kipas angin model Maspion 20,5 cm.AS Maspio...",50.67,327828.0
6,7,ELEMEN BODY SAMPING MAGIC COM PEMANAS RICE COO...,28.49,324898.0
7,8,GEARBOX KIPAS ANGIN COSMOS / SEKAI / MIYAKO / ...,33.95,281823.0
8,9,"KAPASITOR 1,5UF, 2UF, 2,5UF, 3UF, 3,5UF MIKRO ...",54.72,270901.0
9,10,KAPASITOR POMPA AIR 25UF 450VAC CAPASITOR BULE...,9.34,253591.0


[AFTER] Overlap dengan aktual -- ARIMA: 2/10 | LSTM: 4/10


## Sel 17 -- Prediksi Bulan Depan (Semua Produk, Output Operasional Sistem)

Ini adalah output riil yang ditampilkan ke pemilik toko. Dengan CAP_FACTOR_ARIMA=0.8
dan CAP_FACTOR_LSTM=1.0, kedua prediksi kini dibatasi lebih ketat terhadap rekor
penjualan tertinggi historis masing-masing produk dibanding v5.

In [19]:
bulan_depan = bulan_list[-1] + 1
print(f"[BEFORE] Bulan terakhir pada data historis : {bulan_list[-1]}")
print(f"[BEFORE] Bulan yang akan diprediksi (belum ada data aktual) : {bulan_depan}")
print(f"[BEFORE] Jumlah produk yang akan diprediksi : {len(produk_layak)}")
print()

df_a_depan = prediksi_arima(bulan_depan, monthly_all, produk_layak, best_orders_arima, bias_correction, harga_rata2, max_qty_produk)
df_l_depan = prediksi_lstm(bulan_depan, monthly_all, produk_layak, cluster_models, cluster_scalers, cluster_bias,
                            produk_cluster, le, harga_rata2, median_qty_produk, max_qty_produk)
df_depan = pd.merge(df_a_depan, df_l_depan, on='nama_produk', how='outer').fillna(0)
df_depan = df_depan.sort_values('pred_rev_lstm', ascending=False).reset_index(drop=True)
df_depan.insert(0, 'Peringkat', range(1, len(df_depan)+1))

print(f"[AFTER ] Prediksi selesai untuk {len(df_depan)} produk (v6: CAP_FACTOR_ARIMA=0.8, CAP_FACTOR_LSTM=1.0).")
print(f"[AFTER ] Estimasi total revenue bulan {bulan_depan} (LSTM)  : Rp{df_depan['pred_rev_lstm'].sum():,.0f}")
print(f"[AFTER ] Estimasi total revenue bulan {bulan_depan} (ARIMA) : Rp{df_depan['pred_rev_arima'].sum():,.0f}")
print()
print("[AFTER] Top-10 produk berdasarkan estimasi pendapatan LSTM tertinggi:")
display(df_depan.head(10)[['Peringkat','nama_produk','pred_qty_arima','pred_qty_lstm','pred_rev_arima','pred_rev_lstm']])


[BEFORE] Bulan terakhir pada data historis : 2025-03
[BEFORE] Bulan yang akan diprediksi (belum ada data aktual) : 2025-04
[BEFORE] Jumlah produk yang akan diprediksi : 166

[AFTER ] Prediksi selesai untuk 166 produk (v6: CAP_FACTOR_ARIMA=0.8, CAP_FACTOR_LSTM=1.0).
[AFTER ] Estimasi total revenue bulan 2025-04 (LSTM)  : Rp12,957,516
[AFTER ] Estimasi total revenue bulan 2025-04 (ARIMA) : Rp16,546,249

[AFTER] Top-10 produk berdasarkan estimasi pendapatan LSTM tertinggi:


,Peringkat,nama_produk,pred_qty_arima,pred_qty_lstm,pred_rev_arima,pred_rev_lstm
0,1,"KAPASITOR 1,5UF, 2UF, 2,5UF, 3UF, 3,5UF MIKRO ...",125.71,102.68,602486.0,492121.0
1,2,SEAL PDL 255 PEDROLO SIL POMPA AIR MS 14 PDR K...,46.27,65.78,310573.0,441547.0
2,3,KAPASITOR POMPA AIR 20UF 450VAC CAPASITOR BULE...,12.65,18.37,301398.0,437862.0
3,4,KAPASITOR 50 UF 50UF 450VAC 450 VAC CAPASITOR ...,10.48,8.77,501470.0,419912.0
4,5,PINION PANASONIC NASIONAL PEN STOP TARIKAN GEA...,6.99,104.36,24897.0,371845.0
5,6,"AS kipas angin model Maspion 20,5 cm.AS Maspio...",57.59,51.98,372633.0,336332.0
6,7,ELEMEN BODY SAMPING MAGIC COM PEMANAS RICE COO...,15.98,28.94,182246.0,330038.0
7,8,GEARBOX KIPAS ANGIN COSMOS / SEKAI / MIYAKO / ...,13.38,39.43,111063.0,327324.0
8,9,KAPASITOR POMPA AIR 16UF 450VAC CAPASITOR BULE...,8.32,16.58,152587.0,304273.0
9,10,"KAPASITOR 1,5UF, 2UF, 2,5UF, 3UF, 3,5UF MIKRO ...",74.32,59.32,367939.0,293657.0


## Ringkasan Akhir -- Rekapitulasi Preprocessing + Perbandingan v5 vs v6

Sel ini menutup notebook dengan dua tabel: (1) rekapitulasi preprocessing yang identik
dengan v5 (karena tahap ini tidak dipengaruhi CAP_FACTOR), dan (2) tabel perbandingan
metrik akhir v5 vs v6 yang menjadi bukti dokumentasi perubahan konfigurasi.

In [20]:
rekap = pd.DataFrame({
    'Tahap': [
        '1. Load CSV mentah',
        '2. Parsing tanggal (buang tanggal invalid)',
        '3. Filter status "Pesanan Selesai"',
        '4. Agregasi bulanan per produk (monthly_all)',
        '5. Split Training : Testing (80:20)',
        '6a. Filter riwayat >= MIN_BULAN',
        '6b. Filter aktif 3 bulan terakhir',
        '7. Rekayasa fitur LSTM (dropna + eksklusi anomali)',
    ],
    'Satuan': [
        'baris transaksi', 'baris transaksi', 'baris transaksi',
        'baris (bulan x produk)', 'bulan', 'produk', 'produk', 'baris (mtr_clean)'
    ],
    'Jumlah': [
        len(df_raw), n_sesudah_parsing, n_sesudah_filter_status,
        len(monthly_all), f"{len(bulan_train)} train / {len(bulan_test)} test",
        n_lolos_min_bulan, n_lolos_aktif, len(mtr_clean)
    ],
})
print("[REKAPITULASI PREPROCESSING] Identik dengan v5 (tidak dipengaruhi CAP_FACTOR):")
display(rekap)

print()
perbandingan_v5_v6 = pd.DataFrame({
    'Konfigurasi': ['v5 (CAP_ARIMA=1.0, CAP_LSTM=1.5)', 'v6 (CAP_ARIMA=0.8, CAP_LSTM=1.0)'],
    'MAE ARIMA (Rp)': ['2,467,634', f'{mae_agg_a:,.0f}'],
    'RMSE ARIMA (Rp)': ['2,950,422', f'{rmse_agg_a:,.0f}'],
    'MAE LSTM (Rp)': ['1,327,530', f'{mae_agg_l:,.0f}'],
    'RMSE LSTM (Rp)': ['1,587,740', f'{rmse_agg_l:,.0f}'],
})
print("[PERBANDINGAN AKHIR] v5 vs v6 -- gunakan tabel ini sebagai bukti dokumentasi perubahan konfigurasi:")
display(perbandingan_v5_v6)


[REKAPITULASI PREPROCESSING] Identik dengan v5 (tidak dipengaruhi CAP_FACTOR):


,Tahap,Satuan,Jumlah
0,1. Load CSV mentah,baris transaksi,32689
1,2. Parsing tanggal (buang tanggal invalid),baris transaksi,32575
2,"3. Filter status ""Pesanan Selesai""",baris transaksi,31880
3,4. Agregasi bulanan per produk (monthly_all),baris (bulan x produk),8224
4,5. Split Training : Testing (80:20),bulan,47 train / 12 test
5,6a. Filter riwayat >= MIN_BULAN,produk,192
6,6b. Filter aktif 3 bulan terakhir,produk,166
7,7. Rekayasa fitur LSTM (dropna + eksklusi anom...,baris (mtr_clean),3201



[PERBANDINGAN AKHIR] v5 vs v6 -- gunakan tabel ini sebagai bukti dokumentasi perubahan konfigurasi:


,Konfigurasi,MAE ARIMA (Rp),RMSE ARIMA (Rp),MAE LSTM (Rp),RMSE LSTM (Rp)
0,"v5 (CAP_ARIMA=1.0, CAP_LSTM=1.5)","2,467,634","2,950,422","1,327,530","1,587,740"
1,"v6 (CAP_ARIMA=0.8, CAP_LSTM=1.0)","2,408,276","2,870,852","1,327,530","1,587,740"
